# 关于 B 站张雪峰讨论热度分析

## 项目背景
自驱项目：爬取 B 站与"张雪峰"相关的视频数据，分析哪些 UP 主的内容热度最高、什么时长的视频更受欢迎、工作日与休息日发布的效果差异等。

## 数据说明
- 数据源：bilibili_zhangxuefeng_videos.csv（bvid/aid/title/up_name/view播放量/like点赞/coin投币/favorite收藏/share分享/duration时长/pub_time发布时间等字段）
- 工具：Python + pandas

## 分析流程
1. 数据读取与质量检测（缺失值、重复值）
2. 基础指标汇总（总播放/点赞/投币/评论/弹幕量）
3. UP 主维度分析（视频数量、总播放量 Top）
4. 视频时长分段与播放量关系
5. 互动率与综合得分计算
6. 重点 UP 主透视分析（发布时间、工作日/休息日、投币率）


In [1]:
# ===== 导入库 =====
# pandas：数据分析核心库
# numpy：数值计算库
import pandas as pd
import numpy as np

In [2]:
# ===== 读取数据 =====
# 读取 B 站张雪峰相关视频数据 CSV
# 直接打印全表查看
zxf_data = pd.read_csv('D:/浏览器杂项/bilibili_zhangxuefeng_videos.csv')
print(zxf_data)

              bvid              aid          cid  \
0     BV1EcL36KEtW  116583045275398  38373559916   
1     BV1c1Gb6EENv  116618965293859  38544804612   
2     BV1V3L36bE6y  116582592289758  38370675532   
3     BV1u7Go63EC5  116635977389477  38610076496   
4     BV1f6Gf6AEbw  116636497480381  38612107704   
...            ...              ...          ...   
1876  BV1So4y1u7VR        399383492   1153784809   
1877  BV1UiFxe5ExR  113905837867808  28126349906   
1878  BV1XN411z7Jd        489845291   1237035056   
1879  BV1XUGR63EsC  116634467440341  38601360756   
1880  BV1aRfgBaET9  116103535658901  36182098643   

                                              title            up_mid  \
0                           为什么曾经备受尊敬的张雪峰，现如今被恶搞群嘲？        1180757779   
1               张雪峰老师主题曲《念张师》【Hi-Res百万级录音棚试听】(补档×2)  3546821680957794   
2     跑步机，巧乐兹，雪碧，洛克王国是什么梗，为何张雪峰突然口碑崩塌？《赛博人物志解析-张雪峰》  3546557831973276   
3                                  阻止网友玩张雪峰梗的最有效方法！        1332020095   
4         

In [3]:
# ===== 数据探查 =====
# columns：列名；dtypes：类型；size：元素总数；shape：行列数
print('---------------------------------数据表的大致情况------------------------------')
print('数据表的列名:',zxf_data.columns)
print('-----------------------------------------------------------------------------')
print('数据表的类型:',zxf_data.dtypes)
print('-----------------------------------------------------------------------------')
print('数据表的大小:',zxf_data.size)
print('-----------------------------------------------------------------------------')
print('数据表的形状:',zxf_data.shape)

---------------------------------数据表的大致情况------------------------------
数据表的列名: Index(['bvid', 'aid', 'cid', 'title', 'up_mid', 'up_name', 'pub_time',
       'duration', 'view', 'danmaku', 'reply', 'like', 'coin', 'favorite',
       'share', 'tname', 'tags', 'desc'],
      dtype='str')
-----------------------------------------------------------------------------
数据表的类型: bvid            str
aid           int64
cid           int64
title           str
up_mid        int64
up_name         str
pub_time        str
duration      int64
view          int64
danmaku       int64
reply         int64
like          int64
coin          int64
favorite      int64
share         int64
tname       float64
tags        float64
desc            str
dtype: object
-----------------------------------------------------------------------------
数据表的大小: 33858
-----------------------------------------------------------------------------
数据表的形状: (1881, 18)


In [4]:
# 查看视频标识列（bvid/aid/cid：B站视频唯一ID）
zxf_data[['bvid','aid','cid']]

,bvid,aid,cid
0,BV1EcL36KEtW,116583045275398,38373559916
1,BV1c1Gb6EENv,116618965293859,38544804612
2,BV1V3L36bE6y,116582592289758,38370675532
3,BV1u7Go63EC5,116635977389477,38610076496
4,BV1f6Gf6AEbw,116636497480381,38612107704
...,...,...,...
1876,BV1So4y1u7VR,399383492,1153784809
1877,BV1UiFxe5ExR,113905837867808,28126349906
1878,BV1XN411z7Jd,489845291,1237035056
1879,BV1XUGR63EsC,116634467440341,38601360756


In [5]:
# 查看标题与 UP 主信息列
zxf_data[['title','up_mid','up_name']]

,title,up_mid,up_name
0,为什么曾经备受尊敬的张雪峰，现如今被恶搞群嘲？,1180757779,抽象名人堂
1,张雪峰老师主题曲《念张师》【Hi-Res百万级录音棚试听】(补档×2),3546821680957794,不过审的小周qwq
2,跑步机，巧乐兹，雪碧，洛克王国是什么梗，为何张雪峰突然口碑崩塌？《赛博人物志解析-张雪峰》,3546557831973276,木板乱剪
3,阻止网友玩张雪峰梗的最有效方法！,1332020095,VanDebussy德彪西
4,百度地图也爱张雪峰老师,1878613242,麦块B刺猬
...,...,...,...
1876,【张雪峰】人生建议：别学新闻！,476525925,研途考研教育
1877,考公其实没有那么卷，张雪峰太敢说了,3493289978235628,账号已注销
1878,张雪峰劝男孩这段值得反复听，要经历多少，才能释怀以前的往事！,43887712,爱睡觉的_Koala
1879,张雪峰科比小曲合体播放双重暴击！这下听懂了。。,476010056,整活带师兄


In [6]:
# ===== 缺失值与重复值检测 =====
# 统计各列缺失数量与重复行数
print('--------------------------数据表的缺失值和重复值检测---------------------------------------')
print('数据表的缺失值数量:',zxf_data.isnull().sum())
print('-----------------------------------------------------------------------------')
print('数据表的重复值数量:',zxf_data.duplicated().sum())

--------------------------数据表的缺失值和重复值检测---------------------------------------
数据表的缺失值数量: bvid           0
aid            0
cid            0
title          0
up_mid         0
up_name        0
pub_time       0
duration       0
view           0
danmaku        0
reply          0
like           0
coin           0
favorite       0
share          0
tname       1881
tags        1881
desc         313
dtype: int64
-----------------------------------------------------------------------------
数据表的重复值数量: 720


In [7]:
# ===== 缺失值填充 =====
# tname（分区名）缺失填 0；desc（简介）缺失填'未知'
# ⚠️ 注意：tags 列用 tname 的值填充，疑似笔误（应填 tags 自身的缺失值或直接删除该列）
zxf_data['tname'] = zxf_data['tname'].fillna(0)
zxf_data['tags'] = zxf_data['tname'].fillna(0)
zxf_data['desc'] = zxf_data['desc'].fillna('未知')

In [8]:
# 复查缺失值：已全部处理完毕
zxf_data.isnull().sum()

bvid        0
aid         0
cid         0
title       0
up_mid      0
up_name     0
pub_time    0
duration    0
view        0
danmaku     0
reply       0
like        0
coin        0
favorite    0
share       0
tname       0
tags        0
desc        0
dtype: int64

In [9]:
# 将发布时间 pub_time 转为 datetime 类型，便于后续按日期分析
zxf_data['pub_time'] = pd.to_datetime(zxf_data['pub_time'])

In [10]:
# 删除完全重复的行
zxf_data.drop_duplicates(inplace=True)

In [11]:
# 复查重复值数量（应为 0）
zxf_data.duplicated().sum()

np.int64(0)

In [12]:
# 新增'时长(分钟)'列：duration 秒数 ÷ 60，保留 2 位小数
zxf_data['duration_min'] = (zxf_data['duration']/60).round(2)

In [13]:
# 查看时长列
zxf_data['duration_min']

0       4.42
1       4.50
2       4.88
3       2.37
4       1.43
        ... 
1863    1.80
1865    1.37
1873    0.57
1875    0.92
1879    4.13
Name: duration_min, Length: 1161, dtype: float64

In [14]:
# 查看 UP 主与点赞量
zxf_data[['up_name','like']]

,up_name,like
0,抽象名人堂,79750
1,不过审的小周qwq,25821
2,木板乱剪,7975
3,VanDebussy德彪西,67
4,麦块B刺猬,20
...,...,...
1863,辅布师,723
1865,历史影像故事,9549
1873,宋浩老师在B站的学生,705
1875,美味芳香,23


In [15]:
# 查看当前数据表
zxf_data

,bvid,aid,cid,title,up_mid,up_name,pub_time,duration,view,danmaku,reply,like,coin,favorite,share,tname,tags,desc,duration_min
0,BV1EcL36KEtW,116583045275398,38373559916,为什么曾经备受尊敬的张雪峰，现如今被恶搞群嘲？,1180757779,抽象名人堂,2026-05-16 15:22:15,265,3825504,9745,17469,79750,10120,29592,22841,0.0,0.0,未知,4.42
1,BV1c1Gb6EENv,116618965293859,38544804612,张雪峰老师主题曲《念张师》【Hi-Res百万级录音棚试听】(补档×2),3546821680957794,不过审的小周qwq,2026-05-22 23:39:03,270,1101410,1500,9410,25821,3048,9880,32785,0.0,0.0,本视频仅为音乐分享，无任何违规内容，请审核明鉴！\n已按要求替换视频内容和封面，请审核明鉴！,4.50
2,BV1V3L36bE6y,116582592289758,38370675532,跑步机，巧乐兹，雪碧，洛克王国是什么梗，为何张雪峰突然口碑崩塌？《赛博人物志解析-张雪峰》,3546557831973276,木板乱剪,2026-05-16 13:33:33,293,487789,791,1398,7975,192,2472,2499,0.0,0.0,111,4.88
3,BV1u7Go63EC5,116635977389477,38610076496,阻止网友玩张雪峰梗的最有效方法！,1332020095,VanDebussy德彪西,2026-05-25 23:50:34,142,14833,15,64,67,2,13,17,0.0,0.0,-,2.37
4,BV1f6Gf6AEbw,116636497480381,38612107704,百度地图也爱张雪峰老师,1878613242,麦块B刺猬,2026-05-26 01:58:31,86,737,1,10,20,2,2,4,0.0,0.0,-,1.43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1863,BV1CSo6B8EAD,116474161137460,37865981399,张雪峰自己高考前最后一个月：我不信邪，搬桌子到讲台，一定给孩子看看,359138749,辅布师,2026-04-27 09:52:27,108,79408,45,181,723,46,265,189,0.0,0.0,-,1.80
1865,BV189Q6BFETN,116284360559764,36946445583,张雪峰谈跑马拉松，人生不设限，突破自我,700714840,历史影像故事,2026-03-24 21:24:49,82,351231,294,1430,9549,146,1528,1012,0.0,0.0,-,1.37
1873,BV1UpGi6jELo,116622941492898,38551488879,张雪峰老师说上一本二本有个屁用，清华李老师，复旦马老师，北大王老师都不如郑大张雪峰老师,523831614,宋浩老师在B站的学生,2026-05-23 16:31:15,34,24194,99,86,705,6,67,49,0.0,0.0,-,0.57
1875,BV1aTGn6GE1K,116628998132081,38579079492,张雪峰为什么天天跑步的原因找到了,3706967101016165,美味芳香,2026-05-24 18:10:07,55,1362,1,4,23,0,8,4,0.0,0.0,-,0.92


In [16]:
# ===== 核心指标汇总 =====
# 统计视频总数、总播放量、总点赞/投币/评论/弹幕量
print('视频数量:\n',zxf_data['title'].count())
print('总播放量:\n',zxf_data['view'].sum())
print('总点赞量:\n',zxf_data['like'].sum())
print('总投币量:\n',zxf_data['coin'].sum())
print('总评论量:\n',zxf_data['reply'].sum())
print('总弹幕量:\n',zxf_data['danmaku'].sum())

视频数量:
 1161
总播放量:
 317839661
总点赞量:
 8978479
总投币量:
 915300
总评论量:
 915871
总弹幕量:
 589828


In [17]:
# 平均每个视频的播放量与点赞量（千分位格式化）
print(f"平均每个视频的播放量:{zxf_data['view'].mean().round(2):,}")
print(f"平均每个视频的点赞量:{zxf_data['like'].mean().round(2):,}")

平均每个视频的播放量:273,763.7
平均每个视频的点赞量:7,733.4


In [18]:
# ===== UP 主维度聚合 =====
# 按 UP 主分组：视频数量、总播放量、平均播放量
zxf_data_groupby = zxf_data.groupby('up_name').agg(
    视频数量 = ('aid','count'),
    总播放量 = ('view','sum'),
    平均播放量 = ('view','mean')
)
zxf_data_groupby

,视频数量,总播放量,平均播放量
up_name,,,
-子豪好闲-,1,219185,219185.0
4K修复计划,1,55645,55645.0
AAA美团黄袋鼠,1,463607,463607.0
AI播报员,1,527130,527130.0
BOmennoh,1,6040,6040.0
...,...,...,...
麻辣牛尢面,1,217391,217391.0
黄猫切片,1,186535,186535.0
黑曼波语录,1,4415,4415.0


In [19]:
# 视频数量最多的 3 个 UP 主
zxf_data_groupby.nlargest(3,'视频数量').round(2)

,视频数量,总播放量,平均播放量
up_name,,,
辅布师,71,5063998,71323.92
师说考研考公,60,14385726,239762.10
账号已注销,45,15282776,339617.24


In [20]:
# 总播放量最高的 3 个 UP 主（头部内容生产者）
zxf_data_groupby.nlargest(3,'总播放量').round(2)

,视频数量,总播放量,平均播放量
up_name,,,
账号已注销,45,15282776,339617.24
师说考研考公,60,14385726,239762.10
峰乐一,19,12613928,663890.95


In [21]:
# ===== 视频时长分段 =====
# pd.cut 将时长分为：1分钟以内/1-3分钟/3-10分钟/10分钟以上
zxf_data['时间段'] = pd.cut(zxf_data['duration_min'],
                         bins=[0,1,3,10,float('inf')],
                         labels=['0-1分钟','0-3分钟','3-10分钟','10分钟以上'],
                         right=False
                        )

In [22]:
# 不同时长段的平均播放/点赞/投币量（找出最佳时长区间）
zxf_data_groupby2 = zxf_data.groupby('时间段').agg(
    平均播放量 = ('view','mean'),
    平均点赞量 = ('like','mean'),
    平均投币量 = ('coin','mean')
).round(0)
zxf_data_groupby2

,平均播放量,平均点赞量,平均投币量
时间段,,,
0-1分钟,88500.0,2624.0,62.0
0-3分钟,308710.0,7753.0,423.0
3-10分钟,269182.0,7845.0,621.0
10分钟以上,335511.0,10137.0,1981.0


In [23]:
# 播放量 Top20 的投币与播放量
zxf_data[['coin','view']].nlargest(20,'view')

,coin,view
122,97586,7555868
708,50721,6221719
61,81520,5219966
509,24893,4844055
0,10120,3825504
502,53993,3587083
707,1594,3505816
445,1795,3055184
593,1196,2892589
105,36727,2756687


In [24]:
# 播放量最低的 20 条
zxf_data[['coin','view']].nsmallest(20,'view')

,coin,view
623,0,234
913,8,322
12,2,426
927,2,429
17,2,439
8,1,475
434,4,488
14,0,583
511,0,589
11,2,606


In [25]:
# ===== 播放量五等分 =====
# qcut 按播放量分为 很低/低/中/高/很高 五档
zxf_data['播放量大致区间'] = pd.qcut(zxf_data['view'],q=5,labels=['很低','低','中','高','很高'])
zxf_data['播放量大致区间']

0       很高
1       很高
2       很高
3       很低
4       很低
        ..
1863     低
1865     高
1873     低
1875    很低
1879    很低
Name: 播放量大致区间, Length: 1161, dtype: category
Categories (5, str): ['很低' < '低' < '中' < '高' < '很高']

In [26]:
# 各播放量档位的平均投币量（高播放是否带来高投币）
zxf_data.groupby('播放量大致区间')['coin'].mean()

播放量大致区间
很低       9.858369
低       75.866379
中      116.629310
高      305.594828
很高    3437.267241
Name: coin, dtype: float64

In [27]:
# ===== 互动率计算 =====
# 互动率 = (点赞+投币+收藏+分享) / 播放量
# 衡量视频的互动质量
zxf_data['互动率'] = (zxf_data['like']+zxf_data['coin']+zxf_data['favorite']+zxf_data['share'])/zxf_data['view']

In [28]:
# 全量平均互动率
Average_Engagement_Rate = zxf_data['互动率'].mean()
print(f"平均互动率为:{Average_Engagement_Rate}")

平均互动率为:0.0392692030641969


In [29]:
# 查看播放量与互动率
zxf_data[['view','互动率']]

,view,互动率
0,3825504,0.037198
1,1101410,0.064948
2,487789,0.026934
3,14833,0.006674
4,737,0.037992
...,...,...
1863,79408,0.015401
1865,351231,0.034835
1873,24194,0.034182
1875,1362,0.025698


In [30]:
# 筛选：播放量>1万 但 互动率低于全量平均的'高播放低互动'视频
# ⚠️ 阈值 0.0392692030641969 是硬编码的平均值，建议直接用 Average_Engagement_Rate 变量替代
zxf_data.query('view > 10000 and 互动率 < 0.0392692030641969')[['title','up_name','view','互动率']]

,title,up_name,view,互动率
0,为什么曾经备受尊敬的张雪峰，现如今被恶搞群嘲？,抽象名人堂,3825504,0.037198
2,跑步机，巧乐兹，雪碧，洛克王国是什么梗，为何张雪峰突然口碑崩塌？《赛博人物志解析-张雪峰》,木板乱剪,487789,0.026934
3,阻止网友玩张雪峰梗的最有效方法！,VanDebussy德彪西,14833,0.006674
13,张雪峰为何风平逆转？地狱梗实现赛博永生，详细讲解前后成因,慕言and发糕,869322,0.034433
22,武亮正面回应巧乐兹，雪碧和黑张雪峰的黑梗,宋浩老师在B站的学生,221197,0.022776
...,...,...,...,...
1861,普通人去上海发展有出路吗？张雪峰分析完醍醐灌顶,升学规划知识精选,27778,0.028944
1862,专升本计算机想考研进大厂，张雪峰说的很明确了,师说考研考公,31462,0.036012
1863,张雪峰自己高考前最后一个月：我不信邪，搬桌子到讲台，一定给孩子看看,辅布师,79408,0.015401
1865,张雪峰谈跑马拉松，人生不设限，突破自我,历史影像故事,351231,0.034835


In [31]:
# ===== 综合得分 =====
# 加权评分：播放30% + 点赞20% + 投币30% + 收藏20%
# 用于综合衡量视频价值
zxf_data['综合得分'] = zxf_data['view']*0.3+zxf_data['like']*0.2+zxf_data['coin']*0.3+zxf_data['favorite']*0.2

In [32]:
# 综合得分 Top10（按 UP 主查看）
zxf_data[['up_name','综合得分']].nlargest(10,'综合得分')

,up_name,综合得分
122,周书G,2489587.0
708,逗比罐头,2028939.0
61,他们的含金量在增加,1680573.8
509,努力丨丶奋斗,1499894.0
0,抽象名人堂,1172555.6
502,暴叔讲留学,1152573.6
707,精神小伙-命,1066052.2
445,峰乐一,948057.5
593,峰乐一,879946.1
105,乒乓老司机Eason,872497.4


In [33]:
# 查看各列数据类型
print(zxf_data.dtypes)

bvid                       str
aid                      int64
cid                      int64
title                      str
up_mid                   int64
up_name                    str
pub_time        datetime64[us]
duration                 int64
view                     int64
danmaku                  int64
reply                    int64
like                     int64
coin                     int64
favorite                 int64
share                    int64
tname                  float64
tags                   float64
desc                       str
duration_min           float64
时间段                   category
播放量大致区间               category
互动率                    float64
综合得分                   float64
dtype: object


In [34]:
# ===== 重点 UP 主筛选 =====
# 筛选讨论度较高的 6 个 UP 主（周书G/逗比罐头/他们的含金量在增加/辅布师/师说考研考公/账号已注销）
zxf_up_data = zxf_data.query('up_name in ["周书G","逗比罐头","他们的含金量在增加","辅布师","师说考研考公","账号已注销"]')

In [35]:
# 透视表：时间段 × UP主 的平均播放量（观察各 UP 的时长偏好）
zxf_up_data_pivot = pd.pivot_table(zxf_up_data,values='view',index='时间段',columns='up_name',aggfunc='mean').fillna(0).astype('int')
zxf_up_data_pivot

up_name,他们的含金量在增加,周书G,师说考研考公,账号已注销,辅布师,逗比罐头
时间段,,,,,,
0-1分钟,0,0,0,0,56207,0
0-3分钟,0,0,343620,291801,78533,0
3-10分钟,338036,0,216274,341168,28345,3571203
10分钟以上,5219966,7555868,447188,362380,220494,0


In [36]:
# 透视表：各 UP 主的平均投币量
zxf_up_data_pivot2 = pd.pivot_table(zxf_up_data,values='coin',index='up_name',aggfunc='mean')
zxf_up_data_pivot2

,coin
up_name,
他们的含金量在增加,20754.750000
周书G,97586.000000
师说考研考公,341.000000
账号已注销,251.666667
辅布师,66.056338
逗比罐头,28356.000000


In [37]:
# 从发布时间中提取纯日期，新增 publish_date 列
zxf_up_data['publish_date'] = zxf_up_data['pub_time'].dt.date
zxf_up_data['publish_date']

7       2026-05-22
8       2026-05-25
33      2026-05-25
34      2025-10-24
41      2026-05-13
           ...    
1817    2026-05-15
1840    2026-02-24
1851    2026-05-15
1862    2026-03-21
1863    2026-04-27
Name: publish_date, Length: 183, dtype: object

In [38]:
# 透视表：按发布日期的平均播放量（观察时间趋势）
zxf_up_data_pivot3 = pd.pivot_table(zxf_up_data,values='view',index='publish_date',aggfunc='mean').astype('int')
zxf_up_data_pivot3

,view
publish_date,
2021-12-12,7555868
2023-01-18,265354
2023-02-11,228568
2023-03-19,271216
2023-04-05,217344
...,...
2026-05-22,102997
2026-05-23,6902
2026-05-24,1097


In [39]:
# ===== 投币率分析 =====
# 投币率 = 投币量 / 播放量（衡量内容付费意愿）
zxf_up_data['coin_rate'] = zxf_up_data['coin']/zxf_up_data['view']
zxf_up_data[['up_name','coin_rate']]

,up_name,coin_rate
7,辅布师,0.001531
8,辅布师,0.002105
33,辅布师,0.000297
34,师说考研考公,0.000259
41,辅布师,0.002886
...,...,...
1817,辅布师,0.000753
1840,师说考研考公,0.000424
1851,辅布师,0.000753
1862,师说考研考公,0.001367


In [40]:
# 提取发布星期（0=周一 ... 6=周日）
zxf_up_data['weekday_num'] = zxf_up_data['pub_time'].dt.dayofweek
zxf_up_data['weekday_num'].reset_index()

,index,weekday_num
0,7,4
1,8,0
2,33,0
3,34,4
4,41,2
...,...,...
178,1817,4
179,1840,1
180,1851,4
181,1862,5


In [41]:
# 工作日/休息日判断函数（星期≤4 为工作日）
def Date_Judgment(x):
    if x<=4:
        return f"{'工作日'}"
    else:
        return f"{'休息日'}"

In [42]:
# 应用判断，新增 is_workday 列
zxf_up_data['is_workday'] = zxf_up_data['weekday_num'].apply(Date_Judgment)
zxf_up_data[['up_name','is_workday']]

,up_name,is_workday
7,辅布师,工作日
8,辅布师,工作日
33,辅布师,工作日
34,师说考研考公,工作日
41,辅布师,工作日
...,...,...
1817,辅布师,工作日
1840,师说考研考公,工作日
1851,辅布师,工作日
1862,师说考研考公,休息日


In [43]:
# ===== 工作日 vs 休息日投币率对比 =====
# 透视表：时间段 × 工作日/休息日的平均投币率，百分比格式显示
zxf_up_data_pivot5 = pd.pivot_table(zxf_up_data,values='coin_rate',index='时间段',columns='is_workday',aggfunc='mean')
zxf_up_data_pivot5.style.format('{:.2%}')

is_workday,休息日,工作日
时间段,,
0-1分钟,0.11%,0.08%
0-3分钟,0.06%,0.09%
3-10分钟,0.12%,0.09%
10分钟以上,0.34%,0.17%
